# Stage 1 Fine-tuning: NLLB-200 for Literal Translation (AI/ML Domain)

This notebook contains the complete code to fine-tune the `facebook/nllb-200-distilled-600M` model on English-Vietnamese machine translation datasets. Run this notebook on **Kaggle GPU (T4 or P100)**.

In [ ]:
# 1. Install required libraries
!pip install -q transformers datasets accelerate evaluate sacrebleu sentencepiece huggingface_hub

In [ ]:
# 2. Login to Hugging Face Hub
# If you attached HF_TOKEN in Kaggle Secrets, it will log in automatically.
import os
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
if hf_token:
    print("HF_TOKEN found in secrets. Logging in programmatically...")
    login(token=hf_token)
else:
    print("HF_TOKEN not found in environment. Prompting interactive login...")
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
# 3. Load pre-trained model and tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_id = "facebook/nllb-200-distilled-600M"
print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Force target language to Vietnamese during generation
model.config.forced_bos_token_id = tokenizer.convert_tokens_to_ids("vie_Latn")
print(f"Forced BOS Token ID set to Vietnamese: {model.config.forced_bos_token_id}")

In [ ]:
# 4. Prepare and tokenize translation dataset
from datasets import load_dataset

# We load the IWSLT 2015 English-Vietnamese dataset directly from Hugging Face Hub.
# It contains ~133k high-quality sentence pairs.
try:
    raw_datasets = load_dataset("mt_eng_vietnamese", "iwslt2015-en-vi")
    print("Dataset loaded successfully.")
    print(raw_datasets)
except Exception as e:
    print(f"Dataset loading failed: {e}.")

In [ ]:
# 5. Data preprocessing function
max_input_length = 128
max_target_length = 128
src_lang = "eng_Latn"
tgt_lang = "vie_Latn"

def preprocess_function(examples):
    tokenizer.src_lang = src_lang
    tokenizer.tgt_lang = tgt_lang
    
    # Extract English (en) and Vietnamese (vi) texts from 'translation' dictionary
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["vi"] for ex in examples["translation"]]
    
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize dataset
print("Preprocessing and tokenizing dataset...")
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

In [ ]:
# 6. Define evaluation metric (SacreBLEU)
import evaluate
import numpy as np

metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
        
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    # Replace -100 in the labels as we cannot decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [ ]:
# 7. Configure Seq2Seq training arguments
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="nllb-envi-ai-textbook-stage1",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,  # Enable mixed-precision acceleration on GPU
    push_to_hub=True,  # Automatically upload model checkpoint to HF Hub
    hub_model_id="your_hf_username/nllb-envi-ai-textbook-stage1"  # Update with your HF username
)

In [ ]:
# 8. Setup Trainer and start training
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting training...")
trainer.train()

In [ ]:
# 9. Push final model and tokenizer to Hugging Face Hub
trainer.push_to_hub()